# Week 08 - Neural Networks: Multilayer Perceptrons

**DMML - Data Mining & Machine Learning**  
**Due:** End of Week 09  
**Estimated time:** 3-4 hours

This notebook is self-contained. You will train multilayer perceptrons on the Digits dataset and compare them against simpler classifiers in the numeric benchmark.


## What You Are Building

This week has five required implementation pieces:

1. `split_scale_digits(X, y, test_size, random_state)` - stratified split and scaling.
2. `make_tensor_loaders(X_train, X_test, y_train, y_test, batch_size)` - convert arrays to PyTorch `DataLoader`s.
3. `class MLP(nn.Module)` - define a configurable multilayer perceptron.
4. `train_model(model, train_loader, test_loader, optimizer, criterion, epochs, device)` - train and record learning curves.
5. `make_benchmark_long(results, week, dataset, task_type, target, split)` - append neural-network results to the reusable benchmark format.

The modelling question is deliberately practical: on a small image-like dataset, does an MLP outperform strong classical baselines enough to justify the extra training loop and tuning complexity?


In [ ]:
# Imports - keep this cell stable
import warnings
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.25
BATCH_SIZE = 64
EPOCHS = 25
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
print("Using device:", DEVICE)


## Dataset

Digits contains 8-by-8 grayscale images. For the MLP, each image is represented as a flat vector of 64 pixel features.


In [ ]:
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)
images = digits.images
class_names = [str(i) for i in digits.target_names]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for ax, image, label in zip(axes.ravel(), images[:32], y[:32]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(str(label), fontsize=9)
    ax.axis("off")
plt.suptitle("Sample digit images")
plt.tight_layout()
plt.show()


## Task 1 - Split and Scale Digits

Implement `split_scale_digits(X, y, test_size, random_state)`.

Rules:

- Use a stratified train/test split.
- Fit `StandardScaler` only on the training set.
- Transform train and test features with the fitted scaler.
- Return `(X_train_scaled, X_test_scaled, y_train, y_test, scaler)`.


In [ ]:
def split_scale_digits(
    X,
    y,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """Create a stratified split and scale features without leakage."""
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train.astype(np.float64))
    X_test_scaled = scaler.transform(X_test.astype(np.float64))

    return X_train_scaled, X_test_scaled, y_train.copy(), y_test.copy(), scaler


In [ ]:
# Self-check: Task 1
X_train, X_test, y_train, y_test, scaler = split_scale_digits(X, y)

assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert X_train.shape[1] == X.shape[1]
assert hasattr(scaler, "mean_"), "Return the fitted scaler"
assert set(np.unique(y_train)) == set(np.unique(y))
assert set(np.unique(y_test)) == set(np.unique(y))
assert np.allclose(X_train.mean(axis=0), 0, atol=1e-7)

print("Task 1 passed")


## Classical Baselines

Before training a neural network, establish simple baselines on the same split.


In [ ]:
baseline_models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(max_iter=2000),
    "knn_5": KNeighborsClassifier(n_neighbors=5),
}

baseline_rows = []
for name, model in baseline_models.items():
    start = perf_counter()
    model.fit(X_train, y_train)
    fit_time = perf_counter() - start
    pred = model.predict(X_test)
    baseline_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro"),
        "fit_time_sec": fit_time,
    })

baseline_results = pd.DataFrame(baseline_rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
baseline_results


## Task 2 - Tensor DataLoaders

Implement `make_tensor_loaders(X_train, X_test, y_train, y_test, batch_size)`.

Return `(train_loader, test_loader)`. Convert features to `float32` tensors and labels to `long` tensors.


In [ ]:
def make_tensor_loaders(X_train, X_test, y_train, y_test, batch_size: int = BATCH_SIZE):
    """Convert numpy arrays to PyTorch DataLoaders."""
    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    test_dataset = TensorDataset(
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long),
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


In [ ]:
# Self-check: Task 2
train_loader, test_loader = make_tensor_loaders(X_train, X_test, y_train, y_test)
xb, yb = next(iter(train_loader))

assert isinstance(train_loader, DataLoader)
assert isinstance(test_loader, DataLoader)
assert xb.dtype == torch.float32
assert yb.dtype == torch.long
assert xb.shape[1] == X.shape[1]
assert yb.ndim == 1

print("Task 2 passed")


## Task 3 - Define the MLP

Implement `class MLP(nn.Module)`.

Requirements:

- Constructor arguments: `input_dim`, `hidden_sizes`, `output_dim`, `dropout`.
- Use Linear -> ReLU blocks for each hidden layer.
- Add `nn.Dropout(dropout)` after ReLU only when `dropout > 0`.
- The final layer outputs raw logits. Do not apply softmax in the model.


In [ ]:
class MLP(nn.Module):
    """Configurable multilayer perceptron for flat feature vectors."""

    def __init__(self, input_dim: int, hidden_sizes: list[int], output_dim: int, dropout: float = 0.0):
        super().__init__()
        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_sizes:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


In [ ]:
# Self-check: Task 3
test_model = MLP(input_dim=64, hidden_sizes=[64, 32], output_dim=10, dropout=0.1)
dummy_batch = torch.zeros(8, 64)
out = test_model(dummy_batch)

assert isinstance(test_model, nn.Module)
assert out.shape == (8, 10)
assert not torch.allclose(out.softmax(dim=1), out), "Return logits, not probabilities"

print("Task 3 passed")


## Provided Helper: Evaluation

Read this helper before implementing the training loop. It switches the model to evaluation mode and computes loss, accuracy, and macro-F1.


In [ ]:
def evaluate_torch(model, loader, criterion, device=DEVICE):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_targets.append(yb.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true_eval = np.concatenate(all_targets)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true_eval, y_pred),
        "f1_macro": f1_score(y_true_eval, y_pred, average="macro"),
    }


## Task 4 - Training Loop

Implement `train_model(model, train_loader, test_loader, optimizer, criterion, epochs, device)`.

For each epoch:

- train over all batches;
- record train loss;
- evaluate on the test loader using `evaluate_torch`;
- return a history dataframe with columns `epoch`, `train_loss`, `test_loss`, `test_accuracy`, `test_f1_macro`.

This exercise uses the test set as the monitoring set to keep the notebook compact. In project work, keep a separate validation set for model selection.


In [ ]:
def train_model(model, train_loader, test_loader, optimizer, criterion, epochs: int, device=DEVICE) -> pd.DataFrame:
    """Train a PyTorch model and return epoch-level metrics."""
    model.to(device)
    rows = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * xb.size(0)

        train_loss = total_train_loss / len(train_loader.dataset)
        test_metrics = evaluate_torch(model, test_loader, criterion, device=device)
        rows.append(
            {
                "epoch": epoch,
                "train_loss": float(train_loss),
                "test_loss": float(test_metrics["loss"]),
                "test_accuracy": float(test_metrics["accuracy"]),
                "test_f1_macro": float(test_metrics["f1_macro"]),
            }
        )

    columns = ["epoch", "train_loss", "test_loss", "test_accuracy", "test_f1_macro"]
    return pd.DataFrame(rows, columns=columns)


In [ ]:
# Self-check: Task 4
criterion = nn.CrossEntropyLoss()
small_model = MLP(input_dim=64, hidden_sizes=[32], output_dim=10, dropout=0.0).to(DEVICE)
small_optimizer = torch.optim.Adam(small_model.parameters(), lr=0.001)
small_history = train_model(small_model, train_loader, test_loader, small_optimizer, criterion, epochs=2, device=DEVICE)

expected_cols = ["epoch", "train_loss", "test_loss", "test_accuracy", "test_f1_macro"]
assert isinstance(small_history, pd.DataFrame)
assert list(small_history.columns) == expected_cols
assert len(small_history) == 2
assert small_history["test_accuracy"].between(0, 1).all()
assert small_history["train_loss"].notna().all()

print("Task 4 passed")
small_history


## Train MLP Variants

Train a baseline MLP and a dropout MLP. These are the neural-network rows that will enter the benchmark.


In [ ]:
def train_variant(model_name, hidden_sizes, dropout, lr=0.001, epochs=EPOCHS):
    torch.manual_seed(RANDOM_STATE)
    model = MLP(input_dim=64, hidden_sizes=hidden_sizes, output_dim=10, dropout=dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    start = perf_counter()
    history = train_model(model, train_loader, test_loader, optimizer, criterion, epochs=epochs, device=DEVICE)
    fit_time = perf_counter() - start
    final = history.iloc[-1]
    result = {
        "model": model_name,
        "accuracy": final["test_accuracy"],
        "f1_macro": final["test_f1_macro"],
        "fit_time_sec": fit_time,
    }
    return model, history, result

mlp_model, mlp_history, mlp_result = train_variant("mlp_64_32", [64, 32], dropout=0.0)
dropout_model, dropout_history, dropout_result = train_variant("mlp_64_32_dropout_0.2", [64, 32], dropout=0.2)

mlp_results = pd.DataFrame([mlp_result, dropout_result]).sort_values("f1_macro", ascending=False).reset_index(drop=True)
mlp_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mlp_history["epoch"], mlp_history["test_accuracy"], label="MLP")
axes[0].plot(dropout_history["epoch"], dropout_history["test_accuracy"], label="MLP + dropout")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("test accuracy")
axes[0].set_title("Accuracy curves")
axes[0].legend()

axes[1].plot(mlp_history["epoch"], mlp_history["train_loss"], label="MLP train")
axes[1].plot(mlp_history["epoch"], mlp_history["test_loss"], label="MLP test")
axes[1].plot(dropout_history["epoch"], dropout_history["test_loss"], label="Dropout test")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("loss")
axes[1].set_title("Loss curves")
axes[1].legend()

plt.tight_layout()
plt.show()


## Guided Analysis: Confusion Matrix

Inspect the best MLP variant's mistakes.


In [ ]:
best_nn_model = mlp_model if mlp_result["f1_macro"] >= dropout_result["f1_macro"] else dropout_model
best_nn_name = mlp_result["model"] if mlp_result["f1_macro"] >= dropout_result["f1_macro"] else dropout_result["model"]

best_nn_model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = best_nn_model(xb.to(DEVICE))
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_targets.append(yb.numpy())

nn_pred = np.concatenate(all_preds)
nn_true = np.concatenate(all_targets)
cm = confusion_matrix(nn_true, nn_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix: {best_nn_name}")
plt.tight_layout()
plt.show()


## Task 5 - Benchmark Long Format

Implement `make_benchmark_long(results, week, dataset, task_type, target, split)`.

Convert model results into long benchmark format:

- `week`
- `dataset`
- `task_type`
- `target`
- `model`
- `metric`
- `score`
- `split`
- `notes`

Include numeric metric columns such as `accuracy` and `f1_macro`. Do not include `fit_time_sec` as a metric.


In [ ]:
def make_benchmark_long(
    results: pd.DataFrame,
    week: str,
    dataset: str,
    task_type: str,
    target: str,
    split: str,
) -> pd.DataFrame:
    """Convert classifier results to the cumulative benchmark long format."""
    metric_columns = [column for column in results.columns if column not in ["model", "fit_time_sec"]]
    rows = []

    for _, result in results.iterrows():
        for metric in metric_columns:
            if pd.isna(result[metric]):
                continue
            rows.append(
                {
                    "week": week,
                    "dataset": dataset,
                    "task_type": task_type,
                    "target": target,
                    "model": result["model"],
                    "metric": metric,
                    "score": float(result[metric]),
                    "split": split,
                    "notes": "scaled pixels; classical baselines and MLP variants",
                }
            )

    columns = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
    return pd.DataFrame(rows, columns=columns)


In [ ]:
# Self-check: Task 5
all_results = pd.concat([baseline_results, mlp_results], ignore_index=True)
all_results = all_results.sort_values("f1_macro", ascending=False).reset_index(drop=True)

benchmark_long = make_benchmark_long(
    all_results,
    week="W08",
    dataset="Digits",
    task_type="classification",
    target="digit",
    split="75_25_stratified_random_state_42",
)

expected_cols = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
assert list(benchmark_long.columns) == expected_cols
assert {"accuracy", "f1_macro"}.issubset(set(benchmark_long["metric"]))
assert "fit_time_sec" not in set(benchmark_long["metric"])
assert benchmark_long["score"].between(0, 1).all()

print("Task 5 passed")
benchmark_long.head(10)


## Benchmark Wide View

This is the key comparison: does the MLP improve over the classical baselines on the same dataset and split?


In [ ]:
benchmark_wide = (
    benchmark_long
    .pivot_table(
        index=["dataset", "task_type", "target", "metric", "split"],
        columns="model",
        values="score",
        aggfunc="first",
    )
    .reset_index()
)
benchmark_wide.columns.name = None
benchmark_wide


## Reflection

Answer briefly, but concretely.

1. Did the MLP beat logistic regression and k-NN on Digits? Was the margin large enough to justify the extra machinery?
2. Did dropout help on this small dataset? Use the curves and final benchmark numbers.
3. What part of the PyTorch training loop is most different from sklearn's `.fit()` workflow?


## Challenge Tracks Optional

Choose zero, one, or more.

### Track A - Hidden-Size Sweep
Train MLPs with `[32]`, `[64, 32]`, and `[128, 64]`. Add all variants to the benchmark.

### Track B - Optimizer Comparison
Compare Adam with SGD + momentum. Plot the accuracy curves.

### Track C - Learned Weights
Visualise first-layer weights as 8-by-8 images. Do any neurons resemble digit strokes?


In [ ]:
# Optional challenge workspace
# Your code here
